Welcome to the walkthrough for the clustersim and neural field project!  This will be a step-by-step guide to using the package.  It will cover the basics of what each
function is doing, but for more thorough explanations of each step, see the corresponding documentation in the `walkthroughs` folder.

# Getting Started
First, let's set up our virtual environment and install the necessary packages. Of course there are a few ways you can do this, but here is exactly what I used
# Environment
Below is the code I used to setup my virtual environment
Make and activate virtual environment

```
mamba create -n geom_mesh_net python=3.10
conda activate geom_mesh_net
```

Install dependencies

```
mamba install numpy matplotlib pandas pyvista -c conda-forge
pip install torch torchvision torchaudio
pip install torch_geom
pip install plotlyREa
```


### The Data Factory
First thing that needs to happen is to generate the data. This is done in the `data_factory.py` script.  For the intial
neural field model, we only use one pattern at a time, but the model will be adapted in the future and trained on many
point patterns.  Adjust `n_sims` accordingly.  For more information on this, see the `data_factory.md` file.

In [1]:
from geom_mesh_net.core_functions import clustersim as csim
import numpy as np


In [4]:
# set seed
rng = np.random.default_rng(42)

# define domain size
dim1 = 60
dim2 = 60
dim3 = 60

# create domain
domain = {'x': np.array([0.0, dim1]),
             'y': np.array([0.0, dim2]),
             'z': np.array([0.0, dim3])}
#define intensity (points per unit)
intensity = 1

# now opp dim - use smaller domain to save memory
dim1_opp = 30
dim2_opp = 30
dim3_opp = 30

domain_opp = {'x': np.array([0.0, dim1_opp]),
             'y': np.array([0.0, dim2_opp]),
             'z': np.array([0.0, dim3_opp])}

intensity_opp = 1

# number of simulations
n_sims = 5
# overall percent - keep constant
pcp = 0.1

# uniformly vary rho_c, rho_b, cr, rb
rho_c_vec = rng.uniform(low = pcp*2, high = 1, size = n_sims)
rho_b_vec = rng.uniform(low = 0, high = pcp*0.5, size = n_sims)
cr_vec = rng.uniform(low = 3, high = 15, size = n_sims)
rb_vec = rng.uniform(low = 0, high = 0.5, size = n_sims)

# initialize empty matrix to hold calculated rho_c, rho_b, pcp, and their errors
pattern_stats = np.zeros(shape = [n_sims, 9])
save_prefix = "walkthrough_data/"
# make the data
for rho_c, rho_b, cr, rb, i in zip(rho_c_vec, rho_b_vec, cr_vec, rb_vec, range(n_sims)):
    # make opp
    print(i)
    mat, labels = csim.gen_rand_points(intensity = intensity,
                          dim1 = dim1,
                          dim2 = dim2,
                          dim3 = dim3)
    upp = csim.PointPattern3(mat, domain = domain, labels = labels)
    mat_opp, labels_opp = csim.gen_uniform_points(intensity = intensity_opp,
                                            dim1 = dim1_opp,
                                            dim2 = dim2_opp,
                                            dim3 = dim3_opp)
    opp = csim.PointPattern3(mat_opp, domain = domain_opp, labels = labels_opp)
    clust_pattern, rads, centers = csim.clustersim(opp=opp,
                                                   upp=upp,
                                                   pcp=pcp,
                                                   rho_c=rho_c,
                                                   rho_b=rho_b,
                                                   cr=cr,
                                                   rb=rb,
                                                   cut="buffered",
                                                   buffer_factor=1,
                                                   selection='sampled',
                                                   prob_function='Gaussian_decay')
    # save clusters
    name = save_prefix + "clust_pattern_" + str(i)
    np.savez(name,
             coords=clust_pattern.coords,
             domain = clust_pattern.domain,
             labels=clust_pattern.labels,
             radii=rads, centers=centers)

    pcp_clust = (sum(clust_pattern.labels == 2) + sum(clust_pattern.labels == 3)) / clust_pattern.n_points
    rho_c_clust = sum(clust_pattern.labels == 2) / (sum(clust_pattern.labels == 2) + sum(clust_pattern.labels == 1))
    rho_b_clust = sum(clust_pattern.labels == 3) / ((sum(clust_pattern.labels == 0)) + sum(clust_pattern.labels == 3))
    pcp_perc_error = (pcp - pcp_clust) / pcp
    rho_c_perc_error = (rho_c - rho_c_clust) / rho_c
    rho_b_perc_error = (rho_b - rho_b_clust) / rho_b
    pattern_stats[i, 0] = pcp_clust
    pattern_stats[i, 1] = pcp
    pattern_stats[i, 2] = pcp_perc_error
    pattern_stats[i, 3] = rho_c_clust
    pattern_stats[i, 4] = rho_c
    pattern_stats[i, 5] = rho_c_perc_error
    pattern_stats[i, 6] = rho_b_clust
    pattern_stats[i, 7] = rho_b
    pattern_stats[i, 8] = rho_b_perc_error

    print(f"pcp is {pcp_clust}, expected pcp is {pcp}, pcp percent error is {pcp_perc_error}")
    print(
        f" rho_c is {rho_c_clust}, expected rho_c is {rho_c}, rho_c percent error is {rho_c_perc_error}")
    print(
        f" rho_b is {rho_b_clust}, expected_rho_b is {rho_b}, rho_b percent error is {rho_b_perc_error}")
np.save(save_prefix + "pattern_stats", pattern_stats)


0
Estimated Mean Cluster Volume: 1805.1769
Corrected N_Clusts: 8
Scaling Factor: 22.550423709209024
pcp is 0.09285185185185185, expected pcp is 0.1, pcp percent error is 0.07148148148148153
 rho_c is 0.8191638492619774, expected rho_c is 0.8191648388447708, rho_c percent error is 1.2080386589486324e-06
 rho_b is 0.04964855571851103, expected_rho_b is 0.0487811175818378, rho_b percent error is -0.017782252225320032
1
Estimated Mean Cluster Volume: 14556.2242
Corrected N_Clusts: 2
Scaling Factor: 15.878820133816772
Oversampling: reducing 8 grid points to 2
pcp is 0.14230092592592591, expected pcp is 0.1, pcp percent error is -0.4230092592592591
 rho_c is 0.5741839762611276, expected rho_c is 0.5511027518016419, rho_c percent error is -0.041881889328314034
 rho_b is 0.038705714876127936, expected_rho_b is 0.03805698509951765, rho_b percent error is -0.017046273500485615
2
Estimated Mean Cluster Volume: 5219.1958
Corrected N_Clusts: 3
Scaling Factor: 19.27361855903202
Oversampling: reducin

Now we have point patterns!

In [10]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from geom_mesh_net.core_functions import data_loader as dl


# -----------------------------
# General settings
# -----------------------------

# Number of point patterns you want to train separate models for
n_patterns_to_train = 5

# names to look for
data_file_name = "clust_pattern_"
params_file_name = "pattern_stats"

# thinning prob
probs = 0.1

# voxel size
resolution = 0.5

# indices of values within pattern_stats
pcp_ind = 1
rho_c_ind = 4
rho_b_ind = 7

# path to data
data_prefix = "walkthrough_data/"
params_prefix = "walkthrough_data/"

# which types of points will be thinned
marks = "all"

n_points = None
x_weight = 1
y_weight = 1
z_weight = 1
x_exp = 2
y_exp = 2
z_exp = 2
r_weighted_max = None
r_max_weighted_max_ratio = None
prob_function = "Gaussian_decay"
prob_exp = -3
selection = "sampled"
overlap_prob = "highest"

# training settings
epochs = 1000
learning_rate = 0.001

# folder for saved models
model_dir = "trained_models"
os.makedirs(model_dir, exist_ok=True)


# -----------------------------
# Load full dataset
# -----------------------------

dataset = dl.LoadData(
    size=n_patterns_to_train,
    data_file_name=data_file_name,
    params_file_name=params_file_name,
    probs=probs,
    resolution=resolution,
    pcp_ind=pcp_ind,
    rho_c_ind=rho_c_ind,
    rho_b_ind=rho_b_ind,
    data_prefix=data_prefix,
    params_prefix=params_prefix,
    marks=marks,
    n_points=n_points,
    x_weight=x_weight,
    y_weight=y_weight,
    z_weight=z_weight,
    x_exp=x_exp,
    y_exp=y_exp,
    z_exp=z_exp,
    r_weighted_max=r_weighted_max,
    r_max_weighted_max_ratio=r_max_weighted_max_ratio,
    prob_function=prob_function,
    prob_exp=prob_exp,
    selection=selection,
    overlap_prob=overlap_prob,
)




In [11]:
# -----------------------------
# Train one model per pattern
# -----------------------------

for pattern_ind in range(n_patterns_to_train):
    print(f"\n==============================")
    print(f"Training model for pattern {pattern_ind}")
    print(f"==============================")

    # Create a one-pattern dataset view
    single_pattern_dataset = Subset(dataset, [pattern_ind])

    dataloader = DataLoader(
        single_pattern_dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=dl.point_cloud_collate,
    )

    # Initialize a fresh model for this pattern
    model = dl.ContinuousNeuralField2()
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        for batch in dataloader:
            # unpack batch
            coords, domain, labs, xx, yy, zz, full_upp_probs = batch

            # flatten coords for use in model
            x_flat = xx.flatten()
            y_flat = yy.flatten()
            z_flat = zz.flatten()

            # make matrix of coordinate inputs
            inputs = torch.stack([x_flat, y_flat, z_flat], dim=1).float()

            # flatten target probability field
            targets = full_upp_probs.flatten().unsqueeze(1).float()

            # clear old gradients
            optimizer.zero_grad()

            # forward pass
            preds = model(inputs)

            # compute loss
            loss = loss_fn(preds, targets)

            # backpropagate
            loss.backward()

            # update model weights
            optimizer.step()

        # print occasionally so output is not overwhelming
        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"Pattern {pattern_ind} | Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}")
            print("target min/max:", targets.min().item(), targets.max().item())
            print("pred min/max:", preds.min().item(), preds.max().item())

    # save model for this specific pattern
    model_path = os.path.join(model_dir, f"phase2_model_pattern_{pattern_ind}.pt")
    torch.save(model.state_dict(), model_path)

    print(f"Saved model to: {model_path}")


Training model for pattern 0
Pattern 0 | Epoch [1/1000] | Loss: 2.2530
target min/max: 0.04878111928701401 1.0
pred min/max: 0.5138967633247375 0.9851669669151306
Pattern 0 | Epoch [50/1000] | Loss: 0.3074
target min/max: 0.04878111928701401 1.0
pred min/max: 0.006331483367830515 0.47207337617874146
Pattern 0 | Epoch [100/1000] | Loss: 0.3042
target min/max: 0.04878111928701401 1.0
pred min/max: 0.006591207347810268 0.4528297781944275
Pattern 0 | Epoch [150/1000] | Loss: 0.3010
target min/max: 0.04878111928701401 1.0
pred min/max: 0.006017765961587429 0.3953925371170044
Pattern 0 | Epoch [200/1000] | Loss: 0.2897
target min/max: 0.04878111928701401 1.0
pred min/max: 0.009836443699896336 0.2672220468521118
Pattern 0 | Epoch [250/1000] | Loss: 0.2819
target min/max: 0.04878111928701401 1.0
pred min/max: 0.0032678141724318266 0.31167155504226685
Pattern 0 | Epoch [300/1000] | Loss: 0.2795
target min/max: 0.04878111928701401 1.0
pred min/max: 0.002668908331543207 0.32484009861946106
Patte